# Feature Engineering v2 — Validation Selection and Final 50K SVM Training

Train only on official CIFAR-10 development features. Select representation weights, `C`, and `gamma` on the 10,000-image validation split; require at least 95.5% validation accuracy before fitting the final SVM on all 50,000 development rows. The official test features and labels are not loaded here.

## 1. Imports and reproducibility

In [54]:
import os
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

SEED = 30092026
np.random.seed(SEED)
print("SEED:", SEED)

SEED: 30092026


## 2. Fixed validation search and acceptance gate

In [55]:
COMBINED_PATH = "combined_features_v2.npz"
MODEL_PATH = "cifar10_feature_engineering_svm_final_50k_v2.joblib"
VALIDATION_GATE = 0.955
BLOCK_WEIGHTS = (
    (1.0, 1.0, 1.0),
    (1.25, 0.75, 0.75),
    (1.5, 0.5, 0.5),
    (1.0, 1.0, 0.5),
    (1.0, 0.5, 1.0),
)
C_VALUES = (1, 10, 30)
GAMMA_VALUES = ("scale", 0.0005, 0.001)
print("Validation candidates:", len(BLOCK_WEIGHTS) * len(C_VALUES) * len(GAMMA_VALUES))
print("Validation gate:", f"{VALIDATION_GATE:.1%}")

Validation candidates: 45
Validation gate: 95.5%


## 3. Load development features only

In [56]:
if not os.path.exists(COMBINED_PATH):
    raise FileNotFoundError(f"Run Feature Engineering Combination.ipynb first: {COMBINED_PATH}")
with np.load(COMBINED_PATH, allow_pickle=False) as combined:
    X_train = combined["X_train_combined"].astype(np.float32)
    X_val = combined["X_val_combined"].astype(np.float32)
    y_train = combined["y_train"].astype(np.int64)
    y_val = combined["y_val"].astype(np.int64)
    feature_order = combined["feature_order"].tolist()
    feature_dims = combined["feature_dims"].tolist()
    source_archives = combined["source_archives"].tolist()
    run_id = str(combined["run_id"].item())
    assert int(combined["seed"].item()) == SEED
assert feature_order == ["RGB", "AVG", "NTSC"]
assert feature_dims == [512, 512, 512]
assert X_train.shape == (40000, 1536) and X_val.shape == (10000, 1536)
assert y_train.shape == (40000,) and y_val.shape == (10000,)
assert np.isfinite(X_train).all() and np.isfinite(X_val).all()
print("Training:", X_train.shape, "Validation:", X_val.shape)
print("Sources:", source_archives, "Run ID:", run_id)

Training: (40000, 1536) Validation: (10000, 1536)
Sources: ['rgb_features_v2.npz', 'avg_features.npz', 'ntsc_features_v2.npz'] Run ID: 6d2158a8-9f34-4aa1-99b9-0d72837c66df


## 4. Fit constant-feature removal and scaling on training rows only

Block weights are applied **after** scaling. Applying them before `StandardScaler` would cancel their effect.

In [57]:
selector = VarianceThreshold(threshold=0.0)
X_train_selected = selector.fit_transform(X_train)
X_val_selected = selector.transform(X_val)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_selected).astype(np.float32)
X_val_scaled = scaler.transform(X_val_selected).astype(np.float32)
block_ids = np.repeat(np.arange(3), feature_dims)[selector.get_support()]
print("Constant channels removed:", X_train.shape[1] - X_train_scaled.shape[1])
print("Variable channels:", X_train_scaled.shape[1])
print("Variable channels per block:", np.bincount(block_ids, minlength=3).tolist())

Constant channels removed: 246
Variable channels: 1290
Variable channels per block: [512, 266, 512]


## 5. Select block weights and RBF-SVM parameters on validation data

In [58]:
results = []
best_config = None
best_val_accuracy = -np.inf
for weights in BLOCK_WEIGHTS:
    column_weights = np.asarray(weights, dtype=np.float32)[block_ids]
    weighted_train = np.ascontiguousarray(X_train_scaled * column_weights)
    weighted_val = np.ascontiguousarray(X_val_scaled * column_weights)
    for C in C_VALUES:
        for gamma in GAMMA_VALUES:
            svm = SVC(kernel="rbf", C=C, gamma=gamma, cache_size=2048)
            svm.fit(weighted_train, y_train)
            accuracy = accuracy_score(y_val, svm.predict(weighted_val))
            results.append({"weights": weights, "C": C, "gamma": gamma, "validation_accuracy": accuracy})
            print(f"weights={weights}, C={C}, gamma={gamma}: {accuracy:.4%}")
            if accuracy > best_val_accuracy:
                best_val_accuracy = float(accuracy)
                best_config = {"weights": weights, "C": C, "gamma": gamma}
            del svm
    del weighted_train, weighted_val
results_df = pd.DataFrame(results).sort_values("validation_accuracy", ascending=False)
display(results_df.head(15))
print("Best configuration:", best_config)
print(f"Best validation accuracy: {best_val_accuracy:.4%}")

weights=(1.0, 1.0, 1.0), C=1, gamma=scale: 94.4900%
weights=(1.0, 1.0, 1.0), C=1, gamma=0.0005: 94.4900%
weights=(1.0, 1.0, 1.0), C=1, gamma=0.001: 94.4300%
weights=(1.0, 1.0, 1.0), C=10, gamma=scale: 94.4200%
weights=(1.0, 1.0, 1.0), C=10, gamma=0.0005: 94.4100%
weights=(1.0, 1.0, 1.0), C=10, gamma=0.001: 94.3800%
weights=(1.0, 1.0, 1.0), C=30, gamma=scale: 94.4200%
weights=(1.0, 1.0, 1.0), C=30, gamma=0.0005: 94.4100%
weights=(1.0, 1.0, 1.0), C=30, gamma=0.001: 94.3800%
weights=(1.25, 0.75, 0.75), C=1, gamma=scale: 94.4900%
weights=(1.25, 0.75, 0.75), C=1, gamma=0.0005: 94.4500%
weights=(1.25, 0.75, 0.75), C=1, gamma=0.001: 94.5100%
weights=(1.25, 0.75, 0.75), C=10, gamma=scale: 94.5700%
weights=(1.25, 0.75, 0.75), C=10, gamma=0.0005: 94.4900%
weights=(1.25, 0.75, 0.75), C=10, gamma=0.001: 94.5600%
weights=(1.25, 0.75, 0.75), C=30, gamma=scale: 94.5700%
weights=(1.25, 0.75, 0.75), C=30, gamma=0.0005: 94.4900%
weights=(1.25, 0.75, 0.75), C=30, gamma=0.001: 94.5600%
weights=(1.5, 0.5, 

,weights,C,gamma,validation_accuracy
34,"(1.0, 1.0, 0.5)",30,0.0005,0.9459
30,"(1.0, 1.0, 0.5)",10,scale,0.9458
33,"(1.0, 1.0, 0.5)",30,scale,0.9458
35,"(1.0, 1.0, 0.5)",30,0.001,0.9458
32,"(1.0, 1.0, 0.5)",10,0.001,0.9458
12,"(1.25, 0.75, 0.75)",10,scale,0.9457
15,"(1.25, 0.75, 0.75)",30,scale,0.9457
14,"(1.25, 0.75, 0.75)",10,0.001,0.9456
17,"(1.25, 0.75, 0.75)",30,0.001,0.9456
31,"(1.0, 1.0, 0.5)",10,0.0005,0.9455


Best configuration: {'weights': (1.0, 1.0, 0.5), 'C': 30, 'gamma': 0.0005}
Best validation accuracy: 94.5900%


## 6. Validation gate and final 50K refit

Below 95.5% validation accuracy, the notebook prints the next representation to retrain and skips the final fit and save cells without a traceback. The official test set stays unopened.


In [59]:
gate_passed = best_val_accuracy >= VALIDATION_GATE
if not gate_passed:
    if source_archives[2] == "ntsc_features.npz":
        next_step = "Run NTSC Training.ipynb and NTSC Extraction.ipynb for v2, then rerun Combination and Training."
    elif source_archives[1] == "avg_features.npz":
        next_step = "Run AVG Training.ipynb and AVG Extraction.ipynb for v2, then rerun Combination and Training."
    else:
        next_step = "All three v2 representations have been tried; report the validation shortfall."
    print(f"HOLD — validation {best_val_accuracy:.4%} is below the {VALIDATION_GATE:.1%} gate.")
    print(next_step)
    print("Final 50K fit and model save are skipped; official test remains unopened.")
else:
    X_final = np.concatenate((X_train, X_val), axis=0)
    y_final = np.concatenate((y_train, y_val), axis=0)
    assert X_final.shape == (50000, 1536)
    final_selector = VarianceThreshold(threshold=0.0)
    X_final_selected = final_selector.fit_transform(X_final)
    final_scaler = StandardScaler()
    X_final_scaled = final_scaler.fit_transform(X_final_selected).astype(np.float32)
    final_block_ids = np.repeat(np.arange(3), feature_dims)[final_selector.get_support()]
    final_column_weights = np.asarray(best_config["weights"], dtype=np.float32)[final_block_ids]
    X_final_weighted = np.ascontiguousarray(X_final_scaled * final_column_weights)
    final_svm = SVC(kernel="rbf", C=best_config["C"], gamma=best_config["gamma"], cache_size=2048)
    started = time.time()
    final_svm.fit(X_final_weighted, y_final)
    training_seconds = time.time() - started
    print("Final SVM training rows:", len(y_final))
    print("Final variable channels:", X_final_weighted.shape[1])
    print(f"Final SVM fit time: {training_seconds:.1f} seconds")


HOLD — validation 94.5900% is below the 95.5% gate.
Run AVG Training.ipynb and AVG Extraction.ipynb for v2, then rerun Combination and Training.
Final 50K fit and model save are skipped; official test remains unopened.


## 7. Inspect the fitted final feature composition

In [60]:
if gate_passed:
    selected_per_block = np.bincount(final_block_ids, minlength=3).tolist()
    print("Feature order:", feature_order)
    print("Nonconstant channels per block:", selected_per_block)
    print("Block weights:", best_config["weights"])
    print("SVM C/gamma:", best_config["C"], best_config["gamma"])
else:
    print("Final feature composition unavailable until the validation gate passes.")


Final feature composition unavailable until the validation gate passes.


## 8. Save the final 50K model and preprocessing

In [61]:
if gate_passed:
    artifact = {
        "version": "v2",
        "selector": final_selector,
        "scaler": final_scaler,
        "svm": final_svm,
        "block_weights": tuple(best_config["weights"]),
        "C": best_config["C"],
        "gamma": best_config["gamma"],
        "seed": SEED,
        "feature_order": feature_order,
        "feature_dims": feature_dims,
        "source_archives": source_archives,
        "combined_run_id": run_id,
        "validation_accuracy": best_val_accuracy,
        "validation_gate": VALIDATION_GATE,
        "final_training_samples": len(y_final),
        "training_seconds": training_seconds,
    }
    joblib.dump(artifact, MODEL_PATH)
    print("Saved:", MODEL_PATH)
    print(f"Selection validation accuracy: {best_val_accuracy:.4%}")
else:
    print("No final v2 SVM artifact saved; run the next representation fallback first.")


No final v2 SVM artifact saved; run the next representation fallback first.


# End of fusion training run

If the validation gate passed, the saved final SVM is ready for `Feature Engineering Testing.ipynb`. If it did not pass, follow the printed NTSC or AVG fallback and rerun Combination and Training.
